# Prism — Train color histogram + k-NN — Slot D (classical)
**Ron — second classical method**

Hand-written color histograms (8 bins/channel) + chi-squared k-NN voting
(NumPy only, no sklearn) on CIFAR-10.

### Colab setup
1. Clone the repo: `!git clone https://github.com/JethroIsHere/prism.git`
   (or upload the project folder and adjust `PROJECT_ROOT` below).
2. Run the cells top to bottom. CPU is fine — the whole thing takes a few minutes.


In [1]:
import json, os, sys, time
import numpy as np

# Locate the project root: this notebook lives in ron-color-knn/
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, 'ron-color-knn')):
    PROJECT_ROOT = _cwd
elif os.path.basename(_cwd) == 'ron-color-knn':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = "/content/prism" if os.path.isdir("/content/prism") else "."
print("PROJECT_ROOT:", PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "ron-color-knn"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "shared"))

from color_knn import (color_histograms, knn_search, majority_vote,
                       KNearestNeighbors, confusion_matrix,
                       accuracy_and_macro_f1)
from data_utils import load_canonical_splits, CLASS_NAMES, SEED

RESULTS = os.path.join(PROJECT_ROOT, "ron-color-knn", "results")
os.makedirs(RESULTS, exist_ok=True)

K_CANDIDATES = [1, 3, 5, 10, 20]
K_MAX = max(K_CANDIDATES)
print("feature dim: 24 (8 bins x 3 channels)")


PROJECT_ROOT: /home/hatch/workspace/cv-final-project
feature dim: 24 (8 bins x 3 channels)


In [2]:
# CIFAR-10 download (one-time, ~170MB) — skipped if already present.
import tarfile, urllib.request
_DATA = os.path.join(PROJECT_ROOT, "data")
_READY = os.path.join(_DATA, "cifar-10-batches-py", "test_batch")
os.makedirs(_DATA, exist_ok=True)
if not os.path.isfile(_READY):
    print("Downloading CIFAR-10 (~170MB)...")
    _tgz = os.path.join(_DATA, "cifar-10-python.tar.gz")
    urllib.request.urlretrieve(
        "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", _tgz)
    print("Extracting...")
    with tarfile.open(_tgz, "r:gz") as _tf:
        for _m in _tf.getmembers():
            if _m.isfile():  # manual copy: avoids tar ownership quirks
                _out = os.path.join(_DATA, _m.name)
                os.makedirs(os.path.dirname(_out), exist_ok=True)
                with open(_out, 'wb') as _f:
                    _f.write(_tf.extractfile(_m).read())
    print("CIFAR-10 ready.")
else:
    print("CIFAR-10 already present.")


CIFAR-10 already present.


## 1. Sanity check

The feature + distance code must be tested on synthetic images **before**
it touches the real dataset: a solid-colour image has exactly one bin
at 1.0 per channel, and an image's distance to itself is 0.


In [3]:
print('Running color-histogram sanity checks...')
red = np.zeros((32, 32, 3), dtype=np.uint8); red[:, :, 0] = 255
blue = np.zeros((32, 32, 3), dtype=np.uint8); blue[:, :, 2] = 255
Xsyn = np.stack([red, red, blue])
F = color_histograms(Xsyn)
assert F.shape == (3, 24), F.shape
assert np.isfinite(F).all()
# solid red: R channel bin 7 == 1.0, all other R bins 0; G/B bin 0 == 1.0
assert abs(F[0, 7] - 1.0) < 1e-6 and abs(F[0, 0:7]).max() < 1e-6
assert abs(F[0, 8] - 1.0) < 1e-6 and abs(F[0, 16] - 1.0) < 1e-6
# identical images have chi-squared distance 0; red vs blue > 0
from color_knn import _chi2_block
D = _chi2_block(F, F)
assert abs(D[0, 1]) < 1e-6, 'identical images must have distance 0'
assert D[0, 2] > 0, 'different colours must have distance > 0'
# k=1 on a training point returns itself (self-match)
idx, _ = knn_search(F[:2], F[:1], 1)
assert idx[0, 0] == 0, 'nearest neighbour of a point must be itself'
print("ALL SANITY CHECKS PASSED")


Running color-histogram sanity checks...
ALL SANITY CHECKS PASSED


## 2. Data

CIFAR-10: 50,000 train / 10,000 test images. The canonical split is
45k train / 5k val / 10k test (stratified, seed 0) — the same split
every other Prism method uses.


In [4]:
Xtr_u8, ytr, Xva_u8, yva, Xte_u8, yte = load_canonical_splits(
    os.path.join(PROJECT_ROOT, 'data', 'cifar-10-batches-py'))
X_train = Xtr_u8.astype(np.float32) / 255.0
X_val = Xva_u8.astype(np.float32) / 255.0
X_test = Xte_u8.astype(np.float32) / 255.0
print(f'train {X_train.shape}, val {X_val.shape}, test {X_test.shape}')
log = []


train (45000, 32, 32, 3), val (5000, 32, 32, 3), test (10000, 32, 32, 3)


## 3. Color histogram features

8 bins per channel × 3 channels = **24 numbers** per image, each
channel L1-normalised. No spatial information survives — that is the
whole point of this baseline: colour alone, no shape.


In [5]:
t0 = time.perf_counter()
F_train = color_histograms(X_train)
t_feat_train = time.perf_counter() - t0
print(f"train features {F_train.shape} in {t_feat_train:.1f}s")
F_val = color_histograms(X_val)
F_test = color_histograms(X_test)


train features (45000, 24) in 10.9s


## 4. Choose k on the val split

Distances are computed **once** for the val set; every candidate k
is then read off the same neighbour lists. Ties keep the smaller k.


In [6]:
t0 = time.perf_counter()
val_idx, _ = knn_search(F_train, F_val, K_MAX)
print(f"val search took {time.perf_counter() - t0:.1f}s")
best_k, best_acc = None, -1.0
for k in K_CANDIDATES:
    pred = majority_vote(ytr[val_idx], k)
    acc = float((pred == yva).mean())
    print(f"  k={k:2d}  val accuracy {acc * 100:.2f}%")
    if acc > best_acc:
        best_k, best_acc = k, acc
print(f"Chosen k={best_k} (val accuracy {best_acc * 100:.2f}%)")
clf = KNearestNeighbors(k=best_k)
clf.train_feats, clf.train_labels = F_train, ytr


val search took 93.5s
  k= 1  val accuracy 28.86%
  k= 3  val accuracy 27.88%
  k= 5  val accuracy 29.98%
  k=10  val accuracy 31.58%
  k=20  val accuracy 32.96%
Chosen k=20 (val accuracy 32.96%)


## 5. Evaluation (run once on the test set)


In [7]:
t0 = time.perf_counter()
test_idx, _ = knn_search(F_train, F_test, best_k)
t_search = time.perf_counter() - t0
y_pred = majority_vote(ytr[test_idx], best_k)
ms_per_image = t_search / len(X_test) * 1000
print(f"{ms_per_image:.2f} ms/image (CPU, feature + neighbour search)")

cm = confusion_matrix(yte, y_pred, 10)
acc, macro_f1, per_class_f1 = accuracy_and_macro_f1(cm)
print(f"Top-1 accuracy: {acc*100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
log.append(f"top1_accuracy_pct: {acc*100:.2f}")
log.append(f"macro_f1: {macro_f1:.4f}")

metrics = {
    "method": "Color histogram + k-NN (from scratch, NumPy)",
    "dataset": "CIFAR-10",
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
    "top1_accuracy_pct": round(acc*100, 2),
    "macro_f1": round(macro_f1, 4),
    "per_class_f1": {n: round(float(v), 4) for n, v in zip(CLASS_NAMES, per_class_f1)},
    "runtime_ms_per_image_cpu": round(ms_per_image, 2),
    "parameter_count_millions": 0,
    "training_time_minutes": round(t_feat_train / 60, 2),
    "histogram_params": {"bins_per_channel": 8, "feature_dim": 24,
                       "distance": "chi-squared"},
    "knn_params": {"k_candidates": K_CANDIDATES, "k_chosen": best_k,
                 "k_val_accuracy_pct": round(best_acc * 100, 2)},
}
with open(os.path.join(RESULTS, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS, "confusion_matrix.npy"), cm)
clf.save(os.path.join(RESULTS, "model.npz"))
with open(os.path.join(RESULTS, "training_log.txt"), "w") as f:
    f.write("\n".join(log) + "\n")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Color hist + k-NN confusion matrix (acc {acc*100:.2f}%)')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 'confusion_matrix.png'), dpi=150)
plt.close(fig)
print("Saved: metrics.json, confusion_matrix.png/.npy, model.npz, training_log.txt")


14.20 ms/image (CPU, feature + neighbour search)
Top-1 accuracy: 33.40%
Macro F1: 0.3289
Saved: metrics.json, confusion_matrix.png/.npy, model.npz, training_log.txt
